# 01 · Limpieza, transformación y unión de datos

**Objetivo:** construir el dataset final uniendo dos fuentes:

| Fuente | Contenido | Clave de unión |
|---|---|---|
| Kaggle – *Hotel Booking Demand* | Reservas de un hotel urbano y un resort (2015-2017) | `country` (ISO3) + `arrival_date_year` |
| Banco Mundial – *World Development Indicators* | PIB per cápita, población, inflación, salidas turísticas, región y nivel de renta | `Country Code` + año |

**Salida:** `data/processed/hotel_bookings_wb.csv`

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)

RAW = Path('../data/raw')
WB_DIR = RAW / 'banco_mundial'
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

# Decisiones de limpieza parametrizadas (documentadas en el README)
ELIMINAR_DUPLICADOS = True   # el dataset no tiene ID de reserva: ver sección 2
ADR_MAX = 1000               # umbral de outlier imposible de ADR

## 1. Carga e inspección inicial de las reservas

In [2]:
df = pd.read_csv(RAW / 'hotel_bookings.csv')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head()

Filas: 119,390 | Columnas: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [4]:
nulos = df.isna().sum()
(pd.DataFrame({'nulos': nulos, 'pct': (nulos / len(df) * 100).round(2)})
   .query('nulos > 0').sort_values('nulos', ascending=False))

,nulos,pct
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0


## 2. Duplicados

El dataset **no incluye un identificador de reserva**, así que dos filas idénticas pueden ser la misma reserva registrada dos veces o dos reservas distintas con las mismas características (típico en grupos).

Se eliminan porque es la opción conservadora: evita inflar recuentos e ingresos. Se controla con el parámetro `ELIMINAR_DUPLICADOS`.

In [6]:
n_dup = df.duplicated().sum()
print(f'Filas duplicadas exactas: {n_dup:,} ({n_dup / len(df):.1%})')
if ELIMINAR_DUPLICADOS:
    df = df.drop_duplicates().reset_index(drop=True)
print(f'Filas tras el paso: {len(df):,}')

Filas duplicadas exactas: 31,994 (26.8%)
Filas tras el paso: 87,396


## 3. Tratamiento de nulos

| Columna | Decisión | Motivo |
|---|---|---|
| `children` | 0 | Muy pocos casos; lo razonable es que no hubiera niños |
| `country` | `'UNK'` | Se mantiene la reserva, país desconocido |
| `agent` | `'Sin agente'` | Nulo = reserva sin intermediario; es un ID, no un número |
| `company` | Se sustituye por el indicador binario `has_company` | Casi toda la columna es nula |

In [7]:
df['children'] = df['children'].fillna(0).astype(int)
df['country'] = df['country'].fillna('UNK')
df['agent'] = df['agent'].apply(lambda x: 'Sin agente' if pd.isna(x) else str(int(x)))
df['has_company'] = df['company'].notna().astype(int)
df = df.drop(columns='company')
print('Nulos restantes:', df.isna().sum().sum())

Nulos restantes: 0


## 4. Registros inválidos y outliers

In [8]:
total_huespedes = df['adults'] + df['children'] + df['babies']
print('Reservas sin huéspedes:', (total_huespedes == 0).sum())
print('ADR negativo:', (df['adr'] < 0).sum())
print(f'ADR > {ADR_MAX}:', (df['adr'] > ADR_MAX).sum())
print('ADR = 0 (se mantienen, suelen ser cortesías):', (df['adr'] == 0).sum())

Reservas sin huéspedes: 166
ADR negativo: 1
ADR > 1000: 1
ADR = 0 (se mantienen, suelen ser cortesías): 1778


In [9]:
antes = len(df)
df = df[(total_huespedes > 0) & (df['adr'] >= 0) & (df['adr'] <= ADR_MAX)].reset_index(drop=True)
print(f'Eliminadas: {antes - len(df):,} | Filas: {len(df):,}')

Eliminadas: 168 | Filas: 87,228


## 5. Normalización de categorías

- `meal`: según la documentación del dataset, `Undefined` y `SC` significan lo mismo (sin régimen), así que se unifican.
- `market_segment` / `distribution_channel`: los valores `Undefined` se etiquetan como `Unknown`.

In [10]:
df['meal'] = df['meal'].replace('Undefined', 'SC')
for col in ['market_segment', 'distribution_channel']:
    df[col] = df[col].replace('Undefined', 'Unknown')
df[['meal', 'market_segment', 'distribution_channel']].apply(lambda s: s.value_counts().to_dict())

meal                     {'BB': 67905, 'SC': 9883, 'HB': 9080, 'FB': 360}
market_segment          {'Online TA': 51553, 'Offline TA/TO': 13854, '...
distribution_channel    {'TA/TO': 69027, 'Direct': 12953, 'Corporate':...
dtype: object

## 6. Recodificación de países

Algunos códigos no son ISO3 válidos:

- `TMP` → `TLS`: código antiguo de Timor Oriental.
- `CN` → `CAN`: **supuesto**. China ya aparece como `CHN` y Canadá no aparece nunca con su código ISO3, lo que sugiere que `CN` se usó para Canadá. Se documenta como supuesto en el informe.

In [11]:
print(df['country'].isin(['CN', 'CHN', 'CAN', 'TMP']).groupby(df['country']).sum().loc[lambda s: s > 0])
df['country'] = df['country'].replace({'CN': 'CAN', 'TMP': 'TLS'})

country
CHN     816
CN     1093
TMP       3
Name: country, dtype: int64


## 7. Tipos de datos y variables derivadas

In [12]:
meses = {m: i for i, m in enumerate(
    ['January', 'February', 'March', 'April', 'May', 'June', 'July',
     'August', 'September', 'October', 'November', 'December'], start=1)}

df['arrival_month_num'] = df['arrival_date_month'].map(meses)
df['arrival_date'] = pd.to_datetime(dict(year=df['arrival_date_year'],
                                         month=df['arrival_month_num'],
                                         day=df['arrival_date_day_of_month']))
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])
df['booking_date'] = df['arrival_date'] - pd.to_timedelta(df['lead_time'], unit='D')

df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['is_family'] = ((df['children'] + df['babies']) > 0).astype(int)
df['room_changed'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)
df['is_domestic'] = (df['country'] == 'PRT').astype(int)   # hoteles ubicados en Portugal
df['arrival_weekday'] = df['arrival_date'].dt.day_name()

# Ingreso estimado: ADR x noches. Solo se materializa si la reserva no se canceló
df['revenue_est'] = (df['adr'] * df['total_nights']).round(2)
df['revenue_realized'] = np.where(df['is_canceled'] == 0, df['revenue_est'], 0)

temporadas = {12: 'Invierno', 1: 'Invierno', 2: 'Invierno', 3: 'Primavera', 4: 'Primavera',
              5: 'Primavera', 6: 'Verano', 7: 'Verano', 8: 'Verano', 9: 'Otoño',
              10: 'Otoño', 11: 'Otoño'}
df['season'] = df['arrival_month_num'].map(temporadas)

df['lead_time_group'] = pd.cut(df['lead_time'], bins=[-1, 7, 30, 90, 180, 365, np.inf],
                               labels=['0-7 d', '8-30 d', '31-90 d', '91-180 d', '181-365 d', '>365 d'])
df[['arrival_date', 'booking_date', 'total_nights', 'total_guests', 'revenue_est', 'season', 'lead_time_group']].head()

,arrival_date,booking_date,total_nights,total_guests,revenue_est,season,lead_time_group
0,2015-07-01,2014-07-24,0,2,0.0,Verano,181-365 d
1,2015-07-01,2013-06-24,0,2,0.0,Verano,>365 d
2,2015-07-01,2015-06-24,1,1,75.0,Verano,0-7 d
3,2015-07-01,2015-06-18,1,1,75.0,Verano,8-30 d
4,2015-07-01,2015-06-17,2,2,196.0,Verano,8-30 d


## 8. Banco Mundial: carga y reshape

Los CSV del Banco Mundial vienen en **formato ancho**, con una columna por año desde 1960 y 4 filas de cabecera. Se pasan a **formato largo** (país-año-valor), se filtran 2015-2017 y se pivotan para tener una columna por indicador.

In [13]:
INDICADORES = {
    'NY.GDP.PCAP.CD': 'gdp_per_capita_usd',
    'SP.POP.TOTL': 'population',
    'FP.CPI.TOTL.ZG': 'inflation_pct',
    'ST.INT.DPRT': 'tourism_departures',
}
AÑOS = [str(a) for a in sorted(df['arrival_date_year'].unique())]

def cargar_indicador(codigo, nombre):
    ruta = glob.glob(str(WB_DIR / f'API_{codigo}_*.csv'))[0]
    wide = pd.read_csv(ruta, skiprows=4)
    largo = wide.melt(id_vars=['Country Code'], value_vars=AÑOS,
                      var_name='year', value_name=nombre)
    largo['year'] = largo['year'].astype(int)
    return largo.set_index(['Country Code', 'year'])

wb = pd.concat([cargar_indicador(c, n) for c, n in INDICADORES.items()], axis=1).reset_index()
wb.head()

,Country Code,year,gdp_per_capita_usd,population,inflation_pct,tourism_departures
0,ABW,2015,27458.220154,107906.0,0.474764,NaN
1,AFE,2015,1498.715971,607123269.0,5.098562,NaN
2,AFG,2015,565.569730,33831764.0,-0.661709,NaN
3,AFW,2015,1863.401513,417536152.0,2.130817,NaN
4,AGO,2015,3641.728939,28157798.0,9.355972,NaN


In [14]:
meta = pd.read_csv(glob.glob(str(WB_DIR / 'Metadata_Country_*.csv'))[0])
meta = (meta[meta['Region'].notna()]            # descarta agregados (Mundo, UE, etc.)
        [['Country Code', 'TableName', 'Region', 'IncomeGroup']]
        .rename(columns={'TableName': 'country_name', 'Region': 'region',
                         'IncomeGroup': 'income_group'}))

wb = wb.merge(meta, on='Country Code', how='inner').rename(columns={'Country Code': 'country'})
print(f'Países con datos: {wb["country"].nunique()} | Filas país-año: {len(wb):,}')
wb.isna().sum()

Países con datos: 217 | Filas país-año: 651


country                 0
year                    0
gdp_per_capita_usd     17
population              0
inflation_pct          89
tourism_departures    347
country_name            0
region                  0
income_group            0
dtype: int64

## 9. Unión de las dos fuentes

`left join` para no perder reservas: las de países sin datos en el Banco Mundial quedan con indicadores nulos.

In [15]:
filas_antes = len(df)
final = df.merge(wb, left_on=['country', 'arrival_date_year'],
                 right_on=['country', 'year'], how='left').drop(columns='year')
assert len(final) == filas_antes, 'La unión ha duplicado filas'

sin_match = final['region'].isna()
print(f'Reservas sin datos del Banco Mundial: {sin_match.sum():,} ({sin_match.mean():.2%})')
print(final.loc[sin_match, 'country'].value_counts().to_dict())

Reservas sin datos del Banco Mundial: 508 (0.58%)
{'UNK': 447, 'TWN': 43, 'JEY': 8, 'GGY': 2, 'MYT': 2, 'ATA': 2, 'GLP': 1, 'UMI': 1, 'AIA': 1, 'ATF': 1}


In [16]:
final['country_name'] = final['country_name'].fillna(final['country'].replace({'UNK': 'Desconocido'}))
final['region'] = final['region'].fillna('Sin datos')
final['income_group'] = final['income_group'].fillna('Sin datos')

## 10. Validación y exportación

In [17]:
print(f'Dataset final: {final.shape[0]:,} filas x {final.shape[1]} columnas')
assert final.shape[0] >= 50_000 and final.shape[1] >= 20, 'No cumple los mínimos del proyecto'

nulos = final.isna().sum()
print('Nulos por columna (solo indicadores del BM, esperado):')
print(nulos[nulos > 0])

Dataset final: 87,228 filas x 52 columnas
Nulos por columna (solo indicadores del BM, esperado):
gdp_per_capita_usd     524
population             508
inflation_pct          775
tourism_departures    1982
dtype: int64


In [18]:
# CSV estándar (para Python/GitHub) y XLSX (para el dashboard en Excel, sin problemas de separador decimal)
final.to_csv(PROCESSED / 'hotel_bookings_wb.csv', index=False, encoding='utf-8')

excel = final.copy()
excel['lead_time_group'] = excel['lead_time_group'].astype(str)
for col in ['arrival_date', 'booking_date', 'reservation_status_date']:
    excel[col] = excel[col].dt.date      # fecha sin hora para Excel
excel.to_excel(PROCESSED / 'hotel_bookings_wb.xlsx', index=False, sheet_name='reservas')

print('Exportados CSV y XLSX en', PROCESSED)
final.dtypes

Exportados CSV y XLSX en ../data/processed


hotel                                        str
is_canceled                                int64
lead_time                                  int64
arrival_date_year                          int64
arrival_date_month                           str
arrival_date_week_number                   int64
arrival_date_day_of_month                  int64
stays_in_weekend_nights                    int64
stays_in_week_nights                       int64
adults                                     int64
children                                   int64
babies                                     int64
meal                                         str
country                                      str
market_segment                               str
distribution_channel                         str
is_repeated_guest                          int64
previous_cancellations                     int64
previous_bookings_not_canceled             int64
reserved_room_type                           str
assigned_room_type  